In [1]:
# from datasets import load_dataset, concatenate_datasets

# df = load_dataset("Qwen/ProcessBench")
# df = concatenate_datasets(df.values()).to_pandas()

# df["split"] = df["id"].str.split("-").str[0]
# df["steps_len"] = df["steps"].str.len()
# df["per_step_len"] = df["steps"].apply(lambda x: [len(y) for y in x])

import pandas as pd

df = pd.read_parquet("data/processbench100_length_llm.parquet")

In [2]:
from tqdm.auto import tqdm
from openai import OpenAI
from transformers import AutoTokenizer
from model_utils.io_utils import prepare_input, derive_step_rewards_vllm, prepare_batch_input_for_model

```bash
vllm serve hf_cache/Skywork--Skywork-o1-Open-PRM-Qwen-2.5-7B \
    --host 0.0.0.0 \
    --port 8081 \
    --gpu-memory-utilization 0.8 \
    --enable-prefix-caching \
    --dtype auto \
    --task reward
```

```bash
vllm serve hf_cache/Qwen--Qwen2.5-Math-PRM-7B \
    --host 0.0.0.0 \
    --port 8081 \
    --gpu-memory-utilization 0.8 \
    --enable-prefix-caching \
    --dtype auto \
    --task reward
```

In [3]:
openai_api_key = "EMPTY"
openai_api_base = "http://localhost:8081/v1"
client = OpenAI(
    api_key=openai_api_key,
    base_url=openai_api_base,
)
models = client.models.list()
model = models.data[0].id

In [8]:
batch_size = 4
# steps_col = "verbose_steps"
steps_col = "consise_steps"

In [9]:
tokenizer = AutoTokenizer.from_pretrained(model)

input_ids_all = []
token_mask_all = []
for data in tqdm(df.T.to_dict().values(), total=len(df)):
    input_ids, token_mask = prepare_input(
                            model, 
                            problem=data["problem"], 
                            steps=data[steps_col], 
                            tokenizer=tokenizer,
                            convert_to_list=True
    )
    input_ids_all.append(input_ids)
    token_mask_all.append(token_mask)

  0%|          | 0/100 [00:00<?, ?it/s]

In [10]:
import time
all_rewards = []

# Split into batches
num_samples = len(df)

start_time = time.time()

for start_idx in tqdm(range(0, num_samples, batch_size)):
    end_idx = start_idx + batch_size

    batch_input_ids = input_ids_all[start_idx:end_idx]
    batch_token_masks = token_mask_all[start_idx:end_idx]

    batch_input_ids, batch_token_masks = prepare_batch_input_for_model(batch_input_ids, batch_token_masks, pad_token_id=0)

    # Perform inference for this batch
    batch_logits = client.embeddings.create(
        input=batch_input_ids.cpu().tolist(),
        model=model,
    )

    rewards = derive_step_rewards_vllm(
        model,
        batch_logits,
        batch_token_masks,
        tokenizer
    )

    all_rewards.extend(rewards)

    ## Home GPU sepecific time wait for cooldown
    if time.time()-start_time > 60*3:
        time.sleep(5)
        start_time = time.time()

  0%|          | 0/25 [00:00<?, ?it/s]

In [11]:
# df[f"Qwen2.5-Math-PRM-7B--{steps_col}"] = all_rewards
df[f"Skywork-o1-Open-PRM-Qwen-2.5-7B--{steps_col}"] = all_rewards
df.to_parquet("data/processbench100_length_llm.parquet", index=False)

In [12]:
df

,id,generator,problem,steps,final_answer_correct,label,split,steps_len,per_step_len,Qwen2.5-Math-PRM-7B,Skywork-o1-Open-PRM-Qwen-2.5-7B,incorrect_step,verbose_incorrect_step,consise_incorrect_step,verbose_steps,consise_steps,Qwen2.5-Math-PRM-7B--verbose_steps,Qwen2.5-Math-PRM-7B--consise_steps,Skywork-o1-Open-PRM-Qwen-2.5-7B--verbose_steps,Skywork-o1-Open-PRM-Qwen-2.5-7B--consise_steps
0,math-51,Llama-3.1-8B-Instruct,"A group of $N$ students, where $N < 50$, is on...",[Let's break down the problem step by step. Fi...,False,3,math,9,"[268, 343, 158, 433, 189, 253, 288, 246, 225]","[0.984375, 0.9921875, 0.0255126953125, 0.08593...","[0.5964331462646254, 0.4890764454527261, 0.268...","Now, let's try to find the smallest positive i...","Now, let's delve into the process of discoveri...","We solve $8k - 6j = 2$. With $k = 1$, we get $...",[Let's break down the problem step by step. Fi...,[Let's break down the problem step by step. Fi...,"[0.98828125, 0.9921875, 0.026123046875, 0.0668...","[0.98828125, 0.9921875, 0.0255126953125, 0.065...","[0.5959629393988092, 0.48822239382972066, 0.26...","[0.598781508347227, 0.489991571404057, 0.27048..."
1,olympiadbench-479,Qwen2.5-Math-72B-Instruct,Compute the sum of all positive two-digit fact...,[To find the sum of all positive two-digit fac...,False,3,olympiadbench,7,"[352, 163, 112, 256, 90, 146, 92]","[0.96484375, 0.9921875, 0.99609375, 0.77734375...","[0.07807816514495095, 0.126785173178859, 0.166...","Now, we need to find all two-digit factors of ...",In order to identify all of the two-digit fact...,We need to find two-digit factors of \(2^{32} ...,[To find the sum of all positive two-digit fac...,[To find the sum of all positive two-digit fac...,"[0.96484375, 0.9921875, 0.99609375, 0.05957031...","[0.96484375, 0.9921875, 0.99609375, 0.734375, ...","[0.07807816514495095, 0.126785173178859, 0.165...","[0.07585818002124355, 0.1242130083520534, 0.16..."
2,omnimath-617,Llama-3.1-8B-Instruct,For every positive integer $n$ with prime fact...,[To find the strictly increasing functions \( ...,True,3,omnimath,10,"[480, 392, 349, 398, 321, 351, 393, 234, 375, ...","[0.9140625, 0.81640625, 0.73828125, 0.65625, 0...","[0.2295352131664525, 0.2365162364457076, 0.262...","Fourth, **extending to larger differences**: N...","Fourthly, we will explore the scenario wherein...","For larger differences \(d > 10^{100}\), where...",[To find the strictly increasing functions \( ...,[To find the strictly increasing functions \( ...,"[0.91015625, 0.8203125, 0.73828125, 0.58203125...","[0.91015625, 0.82421875, 0.73828125, 0.6679687...","[0.2295352131664525, 0.2379298916111697, 0.262...","[0.23091976292927177, 0.2379298916111697, 0.26..."
3,omnimath-292,Qwen2.5-72B-Instruct,"Let $A, B, C$, and $D$ be points randomly sele...",[To determine the probability that the six lin...,False,2,omnimath,8,"[303, 316, 288, 343, 358, 225, 185, 210]","[0.99609375, 0.984375, 0.88671875, 0.6796875, ...","[0.16559311363082369, 0.13028542074923347, 0.1...",Consider the unit square with vertices at \((0...,Consider the geometric configuration of a unit...,For the slope between any two points in the un...,[To determine the probability that the six lin...,[To determine the probability that the six lin...,"[0.99609375, 0.984375, 0.2490234375, 0.5820312...","[0.99609375, 0.984375, 0.859375, 0.71875, 0.79...","[0.16559311363082369, 0.12940273159163906, 0.1...","[0.16559311363082369, 0.13028542074923347, 0.1..."
4,olympiadbench-76,Qwen2.5-7B-Instruct,"An integer $n$, with $100 \leq n \leq 999$, is...",[To determine the probability that the sum of ...,False,7,olympiadbench,14,"[326, 236, 83, 319, 218, 165, 172, 101, 260, 1...","[0.9921875, 0.9921875, 1.0, 0.98828125, 0.9882...","[0.4542300530235543, 0.6062653664073742, 0.526...","Similarly, if \( b > 9 \) or \( c > 9 \), the ...","Similarly, if the value of \( b \) exceeds 9 o...","If \( b > 9 \) or \( c > 9 \), then:\n\[ \bino...",[To determine the probability that the sum of ...,[To determ